# Predicting Stellar Class — Modeling PLUS

Default pipeline is **`full_blend`**: a multi-seed RealMLP averaged over folds, plus LightGBM / XGBoost / CatBoost, blended by hill-climb on OOF balanced accuracy and calibrated with per-class multipliers.

Switch `PIPELINE_MODE` to `realmlp_only` for a compact single-model run (no boosters). The two main knobs live in **Setup**: `PIPELINE_MODE` and `MLP_SEEDS`; RealMLP training budget (`epochs`, `patience`, schedule) lives in the **RealMLP training config** cell.

## Setup

Imports, global constants and run switches live here. The two main knobs are `PIPELINE_MODE` and `MLP_SEEDS`.


In [ ]:
import warnings, time, gc, os, sys, math, random, glob, json
warnings.filterwarnings('ignore')
from contextlib import contextmanager
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_class_weight


@contextmanager
def quiet():
    """Redirect C-level stdout/stderr (fd 1 & 2) to devnull (hides native GBM GPU chatter)."""
    devnull = os.open(os.devnull, os.O_WRONLY)
    saved = [os.dup(1), os.dup(2)]
    try:
        os.dup2(devnull, 1); os.dup2(devnull, 2)
        yield
    finally:
        os.dup2(saved[0], 1); os.dup2(saved[1], 2)
        os.close(devnull); os.close(saved[0]); os.close(saved[1])


def banner(title, char='=', width=70):
    """Pretty section banner for structured cell output."""
    line = char * width
    print(f'\n{line}\n{title.center(width)}\n{line}')

SEED = 42
N_SPLITS = 5
TARGET = 'class'
ID = 'id'
CLASSES = ['GALAXY', 'QSO', 'STAR']
NC = len(CLASSES)
n_classes = NC
class_to_int = {c: i for i, c in enumerate(CLASSES)}
int_to_class = {i: c for c, i in class_to_int.items()}

# ---- run switches -------------------------------------------------------
PIPELINE_MODE = 'full_blend'          # 'realmlp_only' | 'full_blend'
VALID_PIPELINE_MODES = {'realmlp_only', 'full_blend'}
assert PIPELINE_MODE in VALID_PIPELINE_MODES

RUN_BOOSTERS = PIPELINE_MODE == 'full_blend'
USE_GPU = True
# External SDSS17 rows are opt-in PER FAMILY (ablation): the prior full_blend run
# fed them to the boosters and they scored BELOW RealMLP (which ignores them), so
# the default now flips them OFF to test whether external data hurt the boosters.
MLP_USE_EXTERNAL = False
BOOSTER_USE_EXTERNAL = False
USE_EXTERNAL = (RUN_BOOSTERS and BOOSTER_USE_EXTERNAL) or MLP_USE_EXTERNAL

# RealMLP architectural variants: different shapes per seed give the blend more
# decorrelation than plain seed averaging on a single architecture.
MLP_VARIANTS = [
    {'seed': 42,   'overrides': {}},                                                       # baseline 3x512
    {'seed': 1337, 'overrides': {'hidden_dims': [768, 768, 768]}},                         # wider
    {'seed': 2024, 'overrides': {'hidden_dims': [512, 512, 512, 256], 'dropout': 0.10}},   # deeper
]
RUN_TABM = True          # second NN backbone (TabM-style) for blend diversity
USE_OPTUNA = True        # time-boxed Optuna tuning of the boosters

# SMOKE_TEST: quick end-to-end validation run that exercises EVERY cell/path
# (tiny epochs, 1 NN variant per backbone, no Optuna) in ~15-20 min. Set back
# to False for the real run once it completes without errors.
SMOKE_TEST = False
if SMOKE_TEST:
    MLP_VARIANTS = MLP_VARIANTS[:1]
    USE_OPTUNA = False
    print('*** SMOKE_TEST ON: 1 MLP variant, Optuna off, tiny epochs (see CONFIG/TabM cells) ***')
MLP_SEEDS = [v['seed'] for v in MLP_VARIANTS]
# -------------------------------------------------------------------------
assert N_SPLITS >= 2
assert len(MLP_VARIANTS) >= 1

banner('STELLAR CLASS — Modeling PLUS')
print(f'  pipeline mode    : {PIPELINE_MODE}')
print(f'  boosters         : {RUN_BOOSTERS}  (external={BOOSTER_USE_EXTERNAL})')
print(f'  RealMLP external : {MLP_USE_EXTERNAL}')
print(f'  load external    : {USE_EXTERNAL}')
print(f'  MLP variants     : ' + ' | '.join(
    f"seed {v['seed']} {v['overrides'].get('hidden_dims', '[512,512,512]')}"
    for v in MLP_VARIANTS))

def find_competition_data_dir():
    if Path('/kaggle/input/competitions/playground-series-s6e6').exists():
        return Path('/kaggle/input/competitions/playground-series-s6e6')
    candidates = [p.parent for p in Path('/kaggle/input').rglob('sample_submission.csv')]
    for candidate in sorted(candidates):
        if (candidate / 'train.csv').exists() and (candidate / 'test.csv').exists():
            return candidate
    raise FileNotFoundError('Competition train/test/sample_submission files were not found under /kaggle/input')

ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    DATA_DIR = find_competition_data_dir()
    OUT_DIR = Path('/kaggle/working')
else:
    DATA_DIR = Path('../../docs/dataset')
    OUT_DIR = Path('../../data_processed')
print(f'  environment      : {"Kaggle" if ON_KAGGLE else "local"}  |  data dir: {DATA_DIR}')


### GPU bootstrap

This cell keeps RealMLP usable on Kaggle P100 sessions by installing a compatible PyTorch build when needed. It also disables booster GPU mode when CUDA is unavailable.


In [ ]:
import subprocess, sys, importlib.metadata

def _gpu_name():
    try:
        return subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            text=True).strip()
    except Exception:
        return ''

_NAME = _gpu_name()
print('Detected GPU:', _NAME or '(none / CPU)')
if 'P100' in _NAME:
    try:
        _TORCH_VERSION = importlib.metadata.version('torch')
    except importlib.metadata.PackageNotFoundError:
        _TORCH_VERSION = ''
    if not _TORCH_VERSION.startswith('2.5.1'):
        print('P100 detected -> installing Pascal-compatible torch (cu121) ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'torch==2.5.1', '--index-url',
                        'https://download.pytorch.org/whl/cu121'], check=False)
    else:
        print('P100 detected -> compatible torch already installed:', _TORCH_VERSION)

import torch
CUDA_OK = False
if torch.cuda.is_available():
    try:
        (torch.zeros(8, device='cuda') + 1).sum().item()
        CUDA_OK = True
        print('CUDA kernel test OK on', torch.cuda.get_device_name(0))
    except Exception as e:
        print('WARNING: CUDA kernel test failed, using CPU:', e)
USE_GPU = USE_GPU and CUDA_OK
print('torch', torch.__version__, '| CUDA_OK', CUDA_OK, '| booster GPU:', USE_GPU)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 1. Data

Load the competition files and reduce obvious `float64` / `int64` memory overhead.


In [ ]:
def reduce_mem(df):
    for c in df.select_dtypes('float64').columns:
        df[c] = df[c].astype('float32')
    for c in df.select_dtypes('int64').columns:
        df[c] = pd.to_numeric(df[c], downcast='integer')
    return df

train = reduce_mem(pd.read_csv(DATA_DIR / 'train.csv'))
test = reduce_mem(pd.read_csv(DATA_DIR / 'test.csv'))
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
required_train_cols = {ID, TARGET, 'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift'}
required_test_cols = required_train_cols - {TARGET}
assert required_train_cols.issubset(train.columns), sorted(required_train_cols - set(train.columns))
assert required_test_cols.issubset(test.columns), sorted(required_test_cols - set(test.columns))
print('train:', train.shape, '| test:', test.shape)
train.head(3)


### Optional external data

External SDSS17 rows are appended only when a selected model uses them. Sentinel photometry rows are removed before concatenation.


In [ ]:
train['is_external'] = 0

if USE_EXTERNAL:
    ext_paths = sorted(Path('/kaggle/input').rglob('star_classification.csv'))
    if not ext_paths:
        local_ext = DATA_DIR.parent / 'external' / 'star_classification.csv'
        if local_ext.exists():
            ext_paths = [local_ext]
    if ext_paths:
        ext = pd.read_csv(ext_paths[0])
        ext[TARGET] = ext['class'].astype(str).str.upper()
        band_ok = ~(ext[['u', 'g', 'r', 'i', 'z']] < -50).any(axis=1)
        n_bad = int((~band_ok).sum())
        ext = ext[band_ok].copy()
        for c in ['spectral_type', 'galaxy_population']:
            if c not in ext.columns:
                ext[c] = np.nan
        common = [c for c in train.columns if c in ext.columns]
        ext = reduce_mem(ext[common].copy())
        ext['is_external'] = 1
        train = pd.concat([train, ext], ignore_index=True)
        print('Appended external rows:', len(ext), '| dropped sentinel rows:', n_bad,
              '| new train:', train.shape)
    else:
        print('External dataset not found — skipping.')


## 2. Shared tabular features

These engineered features are used by the optional booster models: SDSS colors, redshift transforms, anomaly flags and magnitude aggregates.


In [ ]:
num_cols = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']
bands = ['u', 'g', 'r', 'i', 'z']
cat_cols = ['spectral_type', 'galaxy_population']

def add_features(df):
    df = df.copy()
    color_cols = []
    for a in range(len(bands)):
        for b in range(a + 1, len(bands)):
            name = f'{bands[a]}_{bands[b]}'
            df[name] = df[bands[a]] - df[bands[b]]
            color_cols.append(name)
    df['redshift_log1p'] = np.log1p(df['redshift'].clip(lower=-0.999))
    df['is_neg_redshift'] = (df['redshift'] < 0).astype('int8')
    df['is_star_like_z'] = (df['redshift'].abs() < 0.002).astype('int8')
    df['z_clip'] = df['redshift'].clip(-0.01, 7.0)
    df['mag_mean'] = df[bands].mean(axis=1)
    df['mag_std'] = df[bands].std(axis=1)
    df['mag_min'] = df[bands].min(axis=1)
    df['mag_max'] = df[bands].max(axis=1)
    df['mag_range'] = df['mag_max'] - df['mag_min']
    return df, color_cols

train_fe, color_cols = add_features(train)
test_fe, _ = add_features(test)

extra_num = ['redshift_log1p', 'is_neg_redshift', 'is_star_like_z', 'z_clip',
             'mag_mean', 'mag_std', 'mag_min', 'mag_max', 'mag_range']
print('color_cols (%d):' % len(color_cols), color_cols)
print('extra_num (%d):' % len(extra_num), extra_num)


### Booster categorical encodings

LightGBM and XGBoost receive ordinal category codes. CatBoost receives string categorical columns through its native categorical interface.


In [ ]:
spectral_order = {'O/B': 0, 'A/F': 1, 'G/K': 2, 'M': 3}
pop_map = {'Blue_Cloud': 0, 'Red_Sequence': 1}

for df in (train_fe, test_fe):
    df['spectral_type_code'] = df['spectral_type'].map(spectral_order).fillna(-1).astype('int16')
    df['galaxy_population_code'] = df['galaxy_population'].map(pop_map).fillna(-1).astype('int16')
    df['z_x_spectral'] = df['redshift'] * (df['spectral_type_code'] + 1)
    df['z_x_pop'] = df['redshift'] * (df['galaxy_population_code'] + 1)

interaction_cols = ['z_x_spectral', 'z_x_pop']
feature_cols = (num_cols + color_cols + extra_num +
                ['spectral_type_code', 'galaxy_population_code'] + interaction_cols)

for df in (train_fe, test_fe):
    for c in cat_cols:
        df[c + '_cat'] = df[c].astype('object').where(df[c].notna(), 'NA').astype(str)
cat_feature_cols = (num_cols + color_cols + extra_num + interaction_cols +
                    ['spectral_type_cat', 'galaxy_population_cat'])
cb_cat_idx = [cat_feature_cols.index('spectral_type_cat'),
              cat_feature_cols.index('galaxy_population_cat')]

train_fe['target'] = train_fe[TARGET].map(class_to_int).astype('int8')
print('LGBM/XGB features (%d)' % len(feature_cols))
print('CatBoost features (%d), cat idx %s' % (len(cat_feature_cols), cb_cat_idx))
print('null check:', train_fe[feature_cols].isnull().sum().sum(),
      test_fe[feature_cols].isnull().sum().sum())


In [ ]:
# Class-centroid distance features for the boosters (point 1: kNN-style signal).
# Centroids are computed on REAL train rows only; at this scale each centroid
# averages 80k-377k points, so a single row's self-contribution is negligible
# (near-zero leakage). Cheap, leakage-safe proxy for a full kNN block.
from sklearn.preprocessing import StandardScaler
_cbase = ['redshift_log1p', 'u_g', 'g_r', 'r_i', 'i_z', 'u_z',
          'mag_mean', 'mag_std', 'mag_range']
_real_mask = (train_fe['is_external'].values == 0)
_cscaler = StandardScaler().fit(train_fe.loc[_real_mask, _cbase].values)
_Ztr = _cscaler.transform(train_fe[_cbase].values).astype('float32')
_Zte = _cscaler.transform(test_fe[_cbase].values).astype('float32')
_yreal = train_fe['target'].values
centroids = np.stack([_Ztr[_real_mask & (_yreal == ci)].mean(0) for ci in range(NC)])

def _cdist(Z):
    return np.sqrt(((Z[:, None, :] - centroids[None, :, :]) ** 2).sum(2)).astype('float32')

cd_cols = [f'cdist_{c}' for c in CLASSES]
for df, Z in [(train_fe, _Ztr), (test_fe, _Zte)]:
    D = _cdist(Z)
    for j, col in enumerate(cd_cols):
        df[col] = D[:, j]
    Ds = np.sort(D, 1)
    df['cdist_argmin'] = D.argmin(1).astype('int16')
    df['cdist_margin'] = (Ds[:, 1] - Ds[:, 0]).astype('float32')

centroid_feats = cd_cols + ['cdist_argmin', 'cdist_margin']
feature_cols = feature_cols + centroid_feats
cat_feature_cols = cat_feature_cols + centroid_feats
print('centroid feats:', centroid_feats)
print('feature_cols now:', len(feature_cols), '| cat_feature_cols now:', len(cat_feature_cols))


## 3. Cross-validation and logging

All models use the same stratified folds over real training rows. External rows, when enabled, are training-only and never appear in validation.


In [ ]:
y = train_fe['target'].values
is_real = (train_fe['is_external'].values == 0)
is_ext_row = ~is_real

counts = np.bincount(y[is_real], minlength=len(CLASSES))
class_w = counts.sum() / (len(CLASSES) * np.maximum(counts, 1))
sample_w = class_w[y].astype('float32')
print('class counts:', counts, '| class weights:', class_w.round(3))

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
ext_idx = np.where(~is_real)[0]
real_idx = np.where(is_real)[0]
folds = []
for tr_r, va_r in skf.split(real_idx, y[real_idx]):
    tr = np.concatenate([real_idx[tr_r], ext_idx])
    va = real_idx[va_r]
    folds.append((tr, va))
print('folds:', [(len(t), len(v)) for t, v in folds])

X = train_fe[feature_cols]
Xc = train_fe[cat_feature_cols]
Xt = test_fe[feature_cols]
Xtc = test_fe[cat_feature_cols]
n_test = len(test_fe)
NC = len(CLASSES)

ARTIFACT_DIR = OUT_DIR / 'modeling_plus_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
# NN checkpoints live in their own folder so the /kaggle/working root stays clean
# (submission CSVs easy to find). On Kaggle this sits inside working; delete after
# the run if you do not want them in the committed output.
CKPT_DIR = OUT_DIR / 'nn_checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG = []
FOLD_LOG = []

def class_recalls(y_true, pred):
    cm = confusion_matrix(y_true, pred, labels=np.arange(NC))
    denom = np.maximum(cm.sum(axis=1), 1)
    return {f'{cls}_recall': cm[i, i] / denom[i] for i, cls in enumerate(CLASSES)}

def score_row(model, proba, seconds=None, notes=''):
    pred = proba.argmax(1)
    row = {
        'model': model,
        'oof_balanced_accuracy': balanced_accuracy_score(y[real_idx], pred),
        'seconds': seconds,
        'features': len(feature_cols),
        'external_rows': int((~is_real).sum()),
        'use_external': bool(USE_EXTERNAL),
        'use_gpu': bool(USE_GPU),
        'notes': notes,
    }
    row.update(class_recalls(y[real_idx], pred))
    return row

def log_model(model, proba, seconds=None, notes=''):
    row = score_row(model, proba[real_idx], seconds=seconds, notes=notes)
    RUN_LOG.append(row)
    display(pd.DataFrame(RUN_LOG).sort_values('oof_balanced_accuracy', ascending=False))
    return row

def save_probs(name, oof, test_pred):
    np.save(ARTIFACT_DIR / f'oof_{name}.npy', oof[real_idx].astype('float32'))
    np.save(ARTIFACT_DIR / f'test_{name}.npy', test_pred.astype('float32'))

def save_run_logs():
    pd.DataFrame(RUN_LOG).to_csv(ARTIFACT_DIR / 'experiment_summary.csv', index=False)
    pd.DataFrame(FOLD_LOG).to_csv(ARTIFACT_DIR / 'fold_metrics.csv', index=False)


## 4. Optional booster models

The next three cells run only in `full_blend` mode. Each booster now trains on REAL rows only by default (`BOOSTER_USE_EXTERNAL=False`) — an ablation against the prior run where external SDSS rows dragged booster OOF below RealMLP. They also get richer shared features (mag min/max/range, z_clip, redshift x population).

### LightGBM

### Booster tuning (optional Optuna)

Time-boxed Optuna on a stratified subsample picks `lgb_params` / `xgb_params` / `cb_params`. The full-data fold loops below consume them; if Optuna is off or unavailable, strong hand-set params are used.

In [ ]:
# Booster hyperparameters with optional time-boxed Optuna (point 1).
# Search runs on a stratified subsample (fast 3-fold, short rounds); the winning
# params are then used by the full-data fold loops below. Falls back to strong
# hand-set params if optuna is missing or disabled.
OPTUNA_TIME_BUDGET = 1500     # seconds per booster (~25 min)
OPTUNA_SUBSAMPLE = 150_000

lgb_params = dict(objective='multiclass', num_class=NC, metric='multi_logloss',
    learning_rate=0.02, num_leaves=255, max_depth=-1, feature_fraction=0.7,
    bagging_fraction=0.8, bagging_freq=1, min_child_samples=80,
    reg_alpha=1.0, reg_lambda=1.0, n_estimators=6000,
    random_state=SEED, n_jobs=-1, verbose=-1)
xgb_params = dict(objective='multi:softprob', num_class=NC, eval_metric='mlogloss',
    learning_rate=0.02, max_depth=8, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, reg_alpha=1.0, reg_lambda=2.0, n_estimators=6000,
    random_state=SEED, n_jobs=-1, tree_method='hist')
cb_params = dict(loss_function='MultiClass', eval_metric='TotalF1',
    learning_rate=0.04, depth=8, l2_leaf_reg=5.0, iterations=6000,
    random_seed=SEED, verbose=0, auto_class_weights='Balanced')
if USE_GPU:
    lgb_params.update(device='gpu')
    xgb_params.update(device='cuda')
    cb_params.update(task_type='GPU', devices='0')

if RUN_BOOSTERS and USE_OPTUNA:
    try:
        import optuna, lightgbm as lgb, xgboost as xgb
        from catboost import CatBoostClassifier, Pool
        from sklearn.model_selection import StratifiedShuffleSplit
        optuna.logging.set_verbosity(optuna.logging.WARNING)

        n_sub = min(OPTUNA_SUBSAMPLE, len(real_idx))
        sss = StratifiedShuffleSplit(n_splits=1, train_size=n_sub, random_state=SEED)
        _s, _ = next(sss.split(real_idx, y[real_idx]))
        sub_idx = real_idx[_s]
        Xs = X.iloc[sub_idx].reset_index(drop=True)
        Xcs = Xc.iloc[sub_idx].reset_index(drop=True)
        ys = y[sub_idx]
        sw_s = class_w[ys].astype('float32')
        folds3 = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(Xs, ys))

        def _cv_lgb(p):
            ba = []
            for tri, vai in folds3:
                m = lgb.LGBMClassifier(**{**lgb_params, **p, 'n_estimators': 1500})
                with quiet():
                    m.fit(Xs.iloc[tri], ys[tri], sample_weight=sw_s[tri],
                          eval_set=[(Xs.iloc[vai], ys[vai])], eval_metric='multi_logloss',
                          callbacks=[lgb.early_stopping(80, verbose=False), lgb.log_evaluation(0)])
                ba.append(balanced_accuracy_score(ys[vai], m.predict_proba(Xs.iloc[vai]).argmax(1)))
            return float(np.mean(ba))

        def obj_lgb(t):
            return _cv_lgb(dict(
                learning_rate=t.suggest_float('learning_rate', 0.02, 0.1, log=True),
                num_leaves=t.suggest_int('num_leaves', 63, 511),
                feature_fraction=t.suggest_float('feature_fraction', 0.5, 1.0),
                bagging_fraction=t.suggest_float('bagging_fraction', 0.6, 1.0),
                min_child_samples=t.suggest_int('min_child_samples', 20, 200),
                reg_alpha=t.suggest_float('reg_alpha', 1e-3, 10, log=True),
                reg_lambda=t.suggest_float('reg_lambda', 1e-3, 10, log=True)))

        st = optuna.create_study(direction='maximize')
        st.optimize(obj_lgb, timeout=OPTUNA_TIME_BUDGET, show_progress_bar=True)
        lgb_params.update(st.best_params)
        print('LGB best CV %.5f  %s' % (st.best_value, st.best_params))

        def _cv_xgb(p):
            ba = []
            for tri, vai in folds3:
                m = xgb.XGBClassifier(**{**xgb_params, **p, 'n_estimators': 1500},
                                      early_stopping_rounds=80)
                with quiet():
                    m.fit(Xs.iloc[tri], ys[tri], sample_weight=sw_s[tri],
                          eval_set=[(Xs.iloc[vai], ys[vai])], verbose=False)
                ba.append(balanced_accuracy_score(ys[vai], m.predict_proba(Xs.iloc[vai]).argmax(1)))
            return float(np.mean(ba))

        def obj_xgb(t):
            return _cv_xgb(dict(
                learning_rate=t.suggest_float('learning_rate', 0.02, 0.1, log=True),
                max_depth=t.suggest_int('max_depth', 5, 11),
                subsample=t.suggest_float('subsample', 0.6, 1.0),
                colsample_bytree=t.suggest_float('colsample_bytree', 0.5, 1.0),
                min_child_weight=t.suggest_float('min_child_weight', 1.0, 10.0),
                reg_alpha=t.suggest_float('reg_alpha', 1e-3, 10, log=True),
                reg_lambda=t.suggest_float('reg_lambda', 1e-3, 10, log=True)))

        st = optuna.create_study(direction='maximize')
        st.optimize(obj_xgb, timeout=OPTUNA_TIME_BUDGET, show_progress_bar=True)
        xgb_params.update(st.best_params)
        print('XGB best CV %.5f  %s' % (st.best_value, st.best_params))

        def _cv_cb(p):
            ba = []
            for tri, vai in folds3:
                m = CatBoostClassifier(**{**cb_params, **p, 'iterations': 1500})
                with quiet():
                    m.fit(Pool(Xcs.iloc[tri], ys[tri], cat_features=cb_cat_idx),
                          eval_set=Pool(Xcs.iloc[vai], ys[vai], cat_features=cb_cat_idx),
                          early_stopping_rounds=80, use_best_model=True)
                ba.append(balanced_accuracy_score(ys[vai], m.predict_proba(Xcs.iloc[vai]).argmax(1)))
            return float(np.mean(ba))

        def obj_cb(t):
            return _cv_cb(dict(
                learning_rate=t.suggest_float('learning_rate', 0.02, 0.12, log=True),
                depth=t.suggest_int('depth', 6, 10),
                l2_leaf_reg=t.suggest_float('l2_leaf_reg', 1.0, 20.0, log=True)))

        st = optuna.create_study(direction='maximize')
        st.optimize(obj_cb, timeout=OPTUNA_TIME_BUDGET, show_progress_bar=True)
        cb_params.update(st.best_params)
        print('CB best CV %.5f  %s' % (st.best_value, st.best_params))

        del Xs, Xcs, ys, sw_s; gc.collect()
    except Exception as e:
        print('Optuna skipped -> using base params:', repr(e))
else:
    print('Optuna disabled (USE_OPTUNA=%s, RUN_BOOSTERS=%s)' % (USE_OPTUNA, RUN_BOOSTERS))
print('booster params ready: lgb/xgb/cb')


In [ ]:
if RUN_BOOSTERS:
    import lightgbm as lgb
    print('lgbm', lgb.__version__, '| lr=%s num_leaves=%s' %
          (lgb_params.get('learning_rate'), lgb_params.get('num_leaves')))

    def run_lgb():
        oof = np.zeros((len(train_fe), NC), dtype='float32')
        test_pred = np.zeros((n_test, NC), dtype='float32')
        bar = tqdm(folds, desc='LightGBM', unit='fold')
        for f, (tr, va) in enumerate(bar, 1):
            tr_b = tr if BOOSTER_USE_EXTERNAL else tr[~is_ext_row[tr]]
            m = lgb.LGBMClassifier(**lgb_params)
            with quiet():
                m.fit(X.iloc[tr_b], y[tr_b], sample_weight=sample_w[tr_b],
                      eval_set=[(X.iloc[va], y[va])], eval_metric='multi_logloss',
                      callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])
            oof[va] = m.predict_proba(X.iloc[va])
            test_pred += m.predict_proba(Xt) / N_SPLITS
            pred = oof[va].argmax(1)
            ba = balanced_accuracy_score(y[va], pred)
            fold_row = {'model': 'lgb', 'fold': f, 'best_iter': m.best_iteration_,
                        'balanced_accuracy': ba, 'train_rows': len(tr_b), 'valid_rows': len(va)}
            fold_row.update(class_recalls(y[va], pred))
            FOLD_LOG.append(fold_row)
            bar.set_postfix(fold=f, best_iter=m.best_iteration_, BA=f'{ba:.5f}')
            del m; gc.collect()
        return oof, test_pred

    t0 = time.time()
    oof_lgb, test_lgb = run_lgb()
    elapsed = time.time() - t0
    save_probs('lgb', oof_lgb, test_lgb)
    log_model('lgb', oof_lgb, elapsed, notes=f'boosters; external={BOOSTER_USE_EXTERNAL}; tuned={USE_OPTUNA}')
    print('LGBM OOF BA: %.5f  (%.0fs)' % (balanced_accuracy_score(y[real_idx], oof_lgb[real_idx].argmax(1)), elapsed))
else:
    print('Skipping LightGBM: PIPELINE_MODE=' + PIPELINE_MODE)


### XGBoost


In [ ]:
if RUN_BOOSTERS:
    import xgboost as xgb
    print('xgb', xgb.__version__, '| lr=%s max_depth=%s' %
          (xgb_params.get('learning_rate'), xgb_params.get('max_depth')))

    def run_xgb():
        oof = np.zeros((len(train_fe), NC), dtype='float32')
        test_pred = np.zeros((n_test, NC), dtype='float32')
        bar = tqdm(folds, desc='XGBoost', unit='fold')
        for f, (tr, va) in enumerate(bar, 1):
            tr_b = tr if BOOSTER_USE_EXTERNAL else tr[~is_ext_row[tr]]
            m = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=200)
            with quiet():
                m.fit(X.iloc[tr_b], y[tr_b], sample_weight=sample_w[tr_b],
                      eval_set=[(X.iloc[va], y[va])], verbose=False)
            oof[va] = m.predict_proba(X.iloc[va])
            test_pred += m.predict_proba(Xt) / N_SPLITS
            pred = oof[va].argmax(1)
            ba = balanced_accuracy_score(y[va], pred)
            fold_row = {'model': 'xgb', 'fold': f, 'best_iter': m.best_iteration,
                        'balanced_accuracy': ba, 'train_rows': len(tr_b), 'valid_rows': len(va)}
            fold_row.update(class_recalls(y[va], pred))
            FOLD_LOG.append(fold_row)
            bar.set_postfix(fold=f, best_iter=m.best_iteration, BA=f'{ba:.5f}')
            del m; gc.collect()
        return oof, test_pred

    t0 = time.time()
    oof_xgb, test_xgb = run_xgb()
    elapsed = time.time() - t0
    save_probs('xgb', oof_xgb, test_xgb)
    log_model('xgb', oof_xgb, elapsed, notes=f'boosters; external={BOOSTER_USE_EXTERNAL}; tuned={USE_OPTUNA}')
    print('XGB OOF BA: %.5f  (%.0fs)' % (balanced_accuracy_score(y[real_idx], oof_xgb[real_idx].argmax(1)), elapsed))
else:
    print('Skipping XGBoost: PIPELINE_MODE=' + PIPELINE_MODE)


### CatBoost


In [ ]:
if RUN_BOOSTERS:
    from catboost import CatBoostClassifier, Pool
    import catboost
    print('catboost', catboost.__version__, '| lr=%s depth=%s' %
          (cb_params.get('learning_rate'), cb_params.get('depth')))

    def run_cb():
        oof = np.zeros((len(train_fe), NC), dtype='float32')
        test_pred = np.zeros((n_test, NC), dtype='float32')
        test_pool = Pool(Xtc, cat_features=cb_cat_idx)
        bar = tqdm(folds, desc='CatBoost', unit='fold')
        for f, (tr, va) in enumerate(bar, 1):
            tr_b = tr if BOOSTER_USE_EXTERNAL else tr[~is_ext_row[tr]]
            tr_pool = Pool(Xc.iloc[tr_b], y[tr_b], cat_features=cb_cat_idx)
            va_pool = Pool(Xc.iloc[va], y[va], cat_features=cb_cat_idx)
            m = CatBoostClassifier(**cb_params)
            with quiet():
                m.fit(tr_pool, eval_set=va_pool, early_stopping_rounds=200, use_best_model=True)
            oof[va] = m.predict_proba(va_pool)
            test_pred += m.predict_proba(test_pool) / N_SPLITS
            pred = oof[va].argmax(1)
            ba = balanced_accuracy_score(y[va], pred)
            fold_row = {'model': 'cb', 'fold': f, 'best_iter': m.get_best_iteration(),
                        'balanced_accuracy': ba, 'train_rows': len(tr_b), 'valid_rows': len(va)}
            fold_row.update(class_recalls(y[va], pred))
            FOLD_LOG.append(fold_row)
            bar.set_postfix(fold=f, best_iter=m.get_best_iteration(), BA=f'{ba:.5f}')
            del m, tr_pool, va_pool; gc.collect()
        return oof, test_pred

    t0 = time.time()
    oof_cb, test_cb = run_cb()
    elapsed = time.time() - t0
    save_probs('cb', oof_cb, test_cb)
    log_model('cb', oof_cb, elapsed, notes=f'native cats; external={BOOSTER_USE_EXTERNAL}; tuned={USE_OPTUNA}')
    print('CatBoost OOF BA: %.5f  (%.0fs)' % (balanced_accuracy_score(y[real_idx], oof_cb[real_idx].argmax(1)), elapsed))
else:
    print('Skipping CatBoost: PIPELINE_MODE=' + PIPELINE_MODE)


## 5. RealMLP feature design

RealMLP uses a separate feature space: pairwise colors, magnitude aggregates, floor-categorized numeric columns and two interaction categories. The interaction categories are target-encoded inside each fold.


In [ ]:
def build_mlp_features(train_df, test_df):
    BANDS = ['u', 'g', 'r', 'i', 'z']
    def add_feats(df):
        df = df.copy()
        for a, b in [('u','g'),('g','r'),('r','i'),('i','z'),
                     ('u','r'),('u','z'),('g','i'),('g','z'),('r','z'),('u','i')]:
            df[f'{a}_{b}'] = df[a] - df[b]
        M = df[BANDS].values
        df['mag_mean'] = M.mean(1); df['mag_std'] = M.std(1)
        df['mag_min'] = M.min(1); df['mag_max'] = M.max(1)
        z = df['redshift'].values
        df['z_log1p'] = np.log1p(np.clip(z, 0, None))
        df['z_clip'] = np.clip(z, -0.01, 7.0)
        return df

    Xtr = add_feats(train_df); Xte = add_feats(test_df)
    cat_cols = Xtr.select_dtypes(include=['object']).columns.tolist()
    num_cols = Xtr.select_dtypes(exclude=['object']).columns.tolist()
    cmap = {}
    combos = sorted([('alpha_cat_', 'delta_cat_'), ('u_cat_', 'z_cat_')])

    def fe(df, fit):
        for col in cat_cols:
            if fit:
                codes, uniques = df[col].factorize(); cmap[col] = uniques
            else:
                code_map = {c: i for i, c in enumerate(cmap[col])}
                codes = df[col].map(code_map).fillna(-1).astype('int32')
            df[col] = codes; df[col] = df[col].astype('category')
        for col in num_cols:
            cn = f'{col}_cat_'
            if fit:
                codes, uniques = np.floor(df[col]).factorize(); cmap[col] = uniques
            else:
                code_map = {c: i for i, c in enumerate(cmap[col])}
                codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
            df[cn] = codes; df[cn] = df[cn].astype('category')
        combo_names = []
        for cols in combos:
            cn = '_'.join(cols) + '_'; combo_names.append(cn)
            s = df[cols[0]].astype(str)
            for c in cols[1:]:
                s = s + '_' + df[c].astype(str)
            if fit:
                codes, uniques = pd.factorize(s, sort=False); cmap[cn] = uniques
            else:
                code_map = {c: i for i, c in enumerate(cmap[cn])}
                codes = s.map(code_map).fillna(-1).astype('int32')
            df[cn] = codes; df[cn] = df[cn].astype('category')
        new_cat = [c for c in df.columns if c.endswith('_')]
        return df, new_cat, combo_names

    Xtr, new_cat, combo_names = fe(Xtr, True)
    Xte, _, _ = fe(Xte, False)
    all_cat = sorted(cat_cols + new_cat)
    Xtr = Xtr.reindex(sorted(Xtr.columns), axis=1)
    Xte = Xte.reindex(sorted(Xte.columns), axis=1)
    return Xtr, Xte, all_cat, combo_names

mlp_train_raw = train.drop(columns=[c for c in [ID, TARGET, 'is_external'] if c in train.columns])
mlp_test_raw = test.drop(columns=[c for c in [ID] if c in test.columns])
X_mlp, X_mlp_test, mlp_cat_cols, combo_names = build_mlp_features(mlp_train_raw, mlp_test_raw)
print('RealMLP design:', X_mlp.shape, '| cat cols:', len(mlp_cat_cols), '| combos:', combo_names)


### RealMLP architecture

The following modules implement the tabular neural network used by the public RealMLP-style solution: numerical preprocessing, categorical embeddings, periodic numerical embeddings and ensemble-aware linear layers.


In [ ]:
class NumericalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, tfms):
        self._tfms = [t for t in tfms if t in ("median_center","robust_scale","smooth_clip","l2_normalize")]
    def fit(self, X, y=None):
        if "median_center" in self._tfms or "robust_scale" in self._tfms:
            self._median = np.median(X, axis=0)
            q = np.quantile(X,0.75,axis=0) - np.quantile(X,0.25,axis=0)
            zi = q == 0.0
            q[zi] = 0.5*(X.max(0)[zi]-X.min(0)[zi])
            self._iqr = 1.0/(q+1e-30); self._iqr[q==0.0] = 0.0
        return self
    def transform(self, X, y=None):
        X = X.copy().astype(np.float32)
        for t in self._tfms:
            if t=="median_center": X -= self._median[None,:]
            elif t=="robust_scale": X *= self._iqr[None,:]
            elif t=="smooth_clip": X = X/np.sqrt(1+(X/3)**2)
            elif t=="l2_normalize":
                n = np.linalg.norm(X,axis=1,keepdims=True); X /= np.where(n==0,1.0,n)
        return X

class CategoricalFeatureLayer(nn.Module):
    def __init__(self, n_ens, cat_dims, embed_dim=8, onehot_thresh=8, device=None):
        super().__init__()
        self.n_ens=n_ens; self.cat_dims=cat_dims; self.onehot_features=[]
        self.embed_layers=nn.ModuleList(); self._embed_feature_indices=[]
        for i,dim in enumerate(cat_dims):
            if dim<=onehot_thresh: self.onehot_features.append(i)
            else:
                self.embed_layers.append(nn.ModuleList([nn.Embedding(dim,embed_dim) for _ in range(n_ens)]))
                self._embed_feature_indices.append(i)
    def forward(self,x):
        b,n_ens,_=x.shape; feats=[]
        if self.onehot_features:
            ox=x[:,:,self.onehot_features]; od=[self.cat_dims[i] for i in self.onehot_features]
            enc=torch.zeros(b,n_ens,sum(od),device=x.device); st=0
            for idx,dim in enumerate(od):
                enc.scatter_(2, ox[:,:,idx:idx+1].long()+st, 1.0); st+=dim
            feats.append(enc)
        for emb_list,fi in zip(self.embed_layers,self._embed_feature_indices):
            fe=[emb_list[mi](x[:,mi,fi:fi+1].long()) for mi in range(self.n_ens)]
            feats.append(torch.cat(fe,dim=1))
        return torch.cat(feats,dim=2)

class ScalingLayer(nn.Module):
    def __init__(self,n_ens,n_features):
        super().__init__(); self.scale=nn.Parameter(torch.ones(n_ens,n_features))
    def forward(self,x): return x*self.scale[None,:,:]

class NTPLinear(nn.Module):
    def __init__(self,n_ens,in_f,out_f,bias=True):
        super().__init__(); self.in_features=in_f
        self.weight=nn.Parameter(torch.randn(n_ens,in_f,out_f))
        self.bias=nn.Parameter(torch.randn(n_ens,out_f)) if bias else None
    def forward(self,x):
        x=torch.einsum("bki,kio->bko",x,self.weight)/math.sqrt(self.in_features)
        if self.bias is not None: x=x+self.bias
        return x

class PBLDEmbedding(nn.Module):
    def __init__(self,n_ens,n_features,hidden_dim=16,out_dim=4,freq_scale=0.1,activation=nn.GELU):
        super().__init__(); self.out_dim=out_dim
        self.w1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim)*freq_scale)
        self.b1=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim))
        self.w2=nn.Parameter(torch.randn(n_ens,n_features,hidden_dim,out_dim-1)/math.sqrt(hidden_dim))
        self.b2=nn.Parameter(torch.zeros(n_ens,n_features,out_dim-1))
        self.act=activation(); nn.init.uniform_(self.b1,-math.pi,math.pi)
    def forward(self,x):
        periodic=torch.cos(2*math.pi*(x.unsqueeze(-1)*self.w1.unsqueeze(0)+self.b1.unsqueeze(0)))
        transformed=self.act(torch.einsum("bkfh,kfhd->bkfd",periodic,self.w2)+self.b2.unsqueeze(0))
        feat=torch.cat([x.unsqueeze(-1),transformed],dim=-1)
        return feat.flatten(start_dim=2)

class RealMLP(nn.Module):
    def __init__(self,output_dim,cat_dims,n_numerical,cfg):
        super().__init__(); n_ens=cfg["n_ens"]; embed_dim=cfg["embed_dim"]; self.n_ens=n_ens
        self.cate=CategoricalFeatureLayer(n_ens,cat_dims,embed_dim,cfg["onehot_thresh"])
        self.num_embed=PBLDEmbedding(n_ens,n_numerical,cfg["pbld_hidden_dim"],cfg["pbld_out_dim"],cfg["pbld_freq_scale"],cfg["pbld_activation"])
        num_emb=n_numerical*cfg["pbld_out_dim"]
        cat_emb=sum(c if c<=cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        total=num_emb+cat_emb; act=cfg["activation"]; layers=[]
        if cfg["add_front_scale"]: layers.append(ScalingLayer(n_ens,total))
        self._dropout_modules=[]; in_dim=total
        for i,h in enumerate(cfg["hidden_dims"]):
            lin=NTPLinear(n_ens,in_dim,h)
            if i==0: self.first_linear=lin
            d=nn.Dropout(cfg["dropout"]); self._dropout_modules.append(d)
            layers+=[lin,act(),d]; in_dim=h
        self.hidden=nn.Sequential(*layers)
        self.output_layer=NTPLinear(n_ens,in_dim,output_dim)
    def forward(self,x_num,x_cat):
        x_num=x_num.unsqueeze(1).expand(-1,self.n_ens,-1)
        x_cat=x_cat.unsqueeze(1).expand(-1,self.n_ens,-1)
        x=torch.cat([self.num_embed(x_num),self.cate(x_cat)],dim=2)
        return F.softmax(self.output_layer(self.hidden(x)),dim=2)

def apply_schedule(v,p,sched,flat=0.3):
    if sched=="constant": return v
    if sched=="cos": return v*(math.cos(math.pi*p)+1)/2
    if sched=="flat_cos":
        if p<flat: return v
        t=(p-flat)/(1-flat); return v*(math.cos(math.pi*t)+1)/2
    if sched=="flat_anneal":
        if p<flat: return v
        t=(p-flat)/(1-flat); return v*(1-t)
    if sched=="sqrt_cos": return v*math.sqrt((math.cos(math.pi*p)+1)/2)
    if sched=="expm4t": return v*math.exp(-4*p)
    raise ValueError(sched)

def get_parameter_groups(model,p):
    fid=id(model.first_linear.weight); sc,pb,fw,ow,bi=[],[],[],[],[]
    for n,pa in model.named_parameters():
        if "num_embed" in n: pb.append(pa)
        elif "scale" in n: sc.append(pa)
        elif id(pa)==fid: fw.append(pa)
        elif "bias" in n: bi.append(pa)
        else: ow.append(pa)
    LR,WD=p["lr"],p["weight_decay"]
    _g=[
        {"params":sc,"lr":LR*p["lr_scale_mult"],"weight_decay":WD*p["wd_scale_mult"]},
        {"params":pb,"lr":LR*p["pbld_lr_factor"],"weight_decay":WD},
        {"params":fw,"lr":LR*p["first_layer_lr_factor"],"weight_decay":WD*p["first_layer_wd_factor"]},
        {"params":ow,"lr":LR,"weight_decay":WD},
        {"params":bi,"lr":LR*p["lr_bias_mult"],"weight_decay":WD*p["wd_bias_mult"]},
    ]
    return [g for g in _g if len(g["params"])>0]

def smooth_ce_loss(yt,yp,ls=0.0,cw=None):
    nc=yp.size(1); ys=torch.full_like(yp,ls/nc)
    ys.scatter_(1,yt.unsqueeze(1),1.0-ls+ls/nc)
    loss=-(ys*torch.log(yp.clamp(1e-15,1))).sum(1)
    if cw is not None:
        sw=cw[yt]; return (loss*sw).sum()/sw.sum()
    return loss.mean()


### RealMLP sklearn wrapper

The wrapper handles fold-local preprocessing, class weights, checkpoint selection and batched probability inference.


In [ ]:
class RealMLP_TD_Classifier(BaseEstimator):
    def __init__(self, **kw): self.params={**CONFIG, **kw}
    def fit(self, Xtr_df, ytr, Xva_df, yva, cat_col_names=None, ckpt_path="ck.pth", X_test=None, desc=""):
        p=self.params; dev=torch.device(p["device"] if torch.cuda.is_available() else "cpu")
        cat_col_names=cat_col_names or []
        num_col_names=[c for c in Xtr_df.columns if c not in cat_col_names]
        Xtn=Xtr_df[num_col_names].values.astype(np.float32); Xvn=Xva_df[num_col_names].values.astype(np.float32)
        Xtc=Xtr_df[cat_col_names].values.astype(np.int64); Xvc=Xva_df[cat_col_names].values.astype(np.int64)
        y_tr=np.asarray(ytr); y_v=np.asarray(yva)
        self.preprocessor_=NumericalPreprocessor(p["tfms"]).fit(Xtn)
        Xtn=self.preprocessor_.transform(Xtn); Xvn=self.preprocessor_.transform(Xvn)
        self.cat_col_names_=cat_col_names; self.num_col_names_=num_col_names
        if cat_col_names:
            allc=[Xtc,Xvc]
            if X_test is not None: allc.append(X_test[cat_col_names].values.astype(np.int64))
            cat_dims=(np.concatenate(allc,0).max(0)+1).tolist()
        else: cat_dims=[]
        self.cat_dims_=cat_dims
        if cat_dims:
            cm=np.array(cat_dims)-1; Xtc=np.clip(Xtc,0,cm); Xvc=np.clip(Xvc,0,cm)
        classes=np.unique(y_tr); self.classes_=classes
        cw=torch.as_tensor(compute_class_weight("balanced",classes=classes,y=y_tr),dtype=torch.float32,device=dev)
        self.model_=p.get("model_cls", RealMLP)(len(classes),cat_dims,Xtn.shape[1],p).to(dev)
        groups=get_parameter_groups(self.model_,p)
        for g in groups: g["lr_base"]=g["lr"]
        opt=torch.optim.AdamW(groups,betas=(p["mom"],p["sq_mom"]))
        Xtn=torch.as_tensor(Xtn,dtype=torch.float32,device=dev); Xtc=torch.as_tensor(Xtc,dtype=torch.long,device=dev)
        ytt=torch.as_tensor(y_tr,dtype=torch.long,device=dev)
        Xvn=torch.as_tensor(Xvn,dtype=torch.float32,device=dev); Xvc=torch.as_tensor(Xvc,dtype=torch.long,device=dev)
        n_ens=p["n_ens"]; tb=p["train_bs"]; eb=p["eval_bs"]; ep=p["epochs"]
        patience=int(p.get("patience", ep)); total=ep*len(y_tr); order=np.arange(len(y_tr)); nc=len(classes)
        best=-np.inf; best_ep=0; no_improve=0; self.best_val_probs_=None; self.stopped_epoch_=ep
        ebar=tqdm(range(ep), desc=desc or "epochs", unit="ep", leave=False)
        for epoch in ebar:
            self.model_.train()
            for s in range(0,len(y_tr),tb):
                prog=(epoch*len(y_tr)+s)/total; idx=order[s:s+tb]
                for g in opt.param_groups: g["lr"]=apply_schedule(g["lr_base"],prog,p["lr_sched"],p["flat_ratio"])
                opt.zero_grad(); yp=self.model_(Xtn[idx],Xtc[idx])
                ls=apply_schedule(p["ls_eps"],prog,p["ls_eps_sched"],p["flat_ratio"])
                dr=apply_schedule(p["dropout"],prog,p["p_drop_sched"],p["flat_ratio"])
                for dm in self.model_._dropout_modules: dm.p=dr
                loss=smooth_ce_loss(ytt[idx].repeat_interleave(n_ens),yp.reshape(-1,nc),ls=ls,cw=cw)
                loss.backward(); torch.nn.utils.clip_grad_norm_(self.model_.parameters(),p["grad_clip"]); opt.step()
            np.random.shuffle(order)
            self.model_.eval()
            with torch.no_grad():
                vp=np.concatenate([self.model_(Xvn[s:s+eb],Xvc[s:s+eb]).mean(1).cpu().numpy() for s in range(0,len(y_v),eb)],0)
            sc=balanced_accuracy_score(y_v,vp.argmax(1))
            if sc>best+1e-6:
                best=sc; best_ep=epoch+1; self.best_val_probs_=vp.copy()
                torch.save(self.model_.state_dict(),ckpt_path); no_improve=0
            else:
                no_improve+=1
            ebar.set_postfix(ba=f"{sc:.5f}", best=f"{best:.5f}", best_ep=best_ep)
            if p["verbosity"]>=2: ebar.write(f"   epoch {epoch+1}/{ep}  ba={sc:.5f}  best={best:.5f}")
            if no_improve>=patience:
                self.stopped_epoch_=epoch+1
                ebar.set_description((desc or "epochs")+f" [early-stop @{epoch+1}]")
                break
        ebar.close()
        self.model_.load_state_dict(torch.load(ckpt_path)); self.best_score_=best; self._dev=dev
        return self
    def predict_proba(self, X):
        eb=self.params["eval_bs"]
        Xn=self.preprocessor_.transform(X[self.num_col_names_].values.astype(np.float32))
        Xc=np.clip(X[self.cat_col_names_].values.astype(np.int64),0,np.array(self.cat_dims_)-1)
        Xn=torch.as_tensor(Xn,dtype=torch.float32,device=self._dev); Xc=torch.as_tensor(Xc,dtype=torch.long,device=self._dev)
        self.model_.eval()
        with torch.no_grad():
            return np.concatenate([self.model_(Xn[s:s+eb],Xc[s:s+eb]).mean(1).cpu().numpy() for s in range(0,len(Xn),eb)],0)

### RealMLP training config

The config keeps the architecture and optimizer schedule in one place. Adjust `epochs`, batch sizes or hidden dimensions here when experimenting.


In [ ]:
CONFIG = {
    "n_ens": 8, "embed_dim": 7, "onehot_thresh": 10,
    "hidden_dims": [512, 512, 512], "dropout": 0.05, "p_drop_sched": "expm4t",
    "activation": nn.SiLU, "add_front_scale": True,
    "pbld_hidden_dim": 20, "pbld_out_dim": 5, "pbld_freq_scale": 5.0,
    "pbld_activation": nn.PReLU, "pbld_lr_factor": 0.093,
    "lr": 0.01, "mom": 0.9, "sq_mom": 0.98, "lr_sched": "flat_cos", "flat_ratio": 0.4,
    "first_layer_lr_factor": 1.0, "first_layer_wd_factor": 0.1,
    "lr_scale_mult": 10.0, "lr_bias_mult": 0.1, "weight_decay": 0.013,
    "wd_scale_mult": 0.1, "wd_bias_mult": 0.5, "grad_clip": 1.0,
    "ls_eps": 0.04, "ls_eps_sched": "cos",
    "tfms": ["median_center", "robust_scale"],
    # epochs lengthened + flat_ratio raised so the cosine anneal lands the
    # accuracy peak near the END of training instead of mid-run (was peaking
    # at ep 4-5 of 8). `patience` early-stops once the best epoch stops moving,
    # so the longer budget costs little when a fold converges sooner.
    "epochs": 18, "patience": 6, "train_bs": 256, "eval_bs": 10240, "verbosity": 0,
    "device": "cuda" if CUDA_OK else "cpu", "random_state": 42,
}
if SMOKE_TEST:
    CONFIG['epochs'] = 2
    CONFIG['patience'] = 2
FOLDS = N_SPLITS
n_classes = NC
print('RealMLP CONFIG: epochs=%d patience=%d lr_sched=%s flat_ratio=%.2f hidden=%s n_ens=%d'
      % (CONFIG['epochs'], CONFIG['patience'], CONFIG['lr_sched'],
         CONFIG['flat_ratio'], CONFIG['hidden_dims'], CONFIG['n_ens']))

### TabM backbone (#2)

Second NN backbone reusing the RealMLP training infrastructure (preprocessing, schedules, early stopping) but with a plain MLP head and no periodic numerical embeddings. Different nature than RealMLP → genuine decorrelation for the blend/stack.

In [ ]:
# TabM-style backbone #2: plain MLP with BatchEnsemble-like multi-head ensembling
# (NTPLinear gives each of n_ens members its own weights), NO periodic embeddings
# and NO target-encoding-specific tricks beyond the shared feature frame. The lack
# of PBLD periodic embeddings is the key difference from RealMLP -> real decorrelation.
class TabM(nn.Module):
    def __init__(self, output_dim, cat_dims, n_numerical, cfg):
        super().__init__()
        n_ens = cfg["n_ens"]; embed_dim = cfg["embed_dim"]; self.n_ens = n_ens
        self.cate = CategoricalFeatureLayer(n_ens, cat_dims, embed_dim, cfg["onehot_thresh"])
        cat_emb = sum(c if c <= cfg["onehot_thresh"] else embed_dim for c in cat_dims)
        total = n_numerical + cat_emb
        act = cfg["activation"]; layers = []
        if cfg["add_front_scale"]:
            layers.append(ScalingLayer(n_ens, total))
        self._dropout_modules = []; in_dim = total
        for i, h in enumerate(cfg["hidden_dims"]):
            lin = NTPLinear(n_ens, in_dim, h)
            if i == 0:
                self.first_linear = lin
            d = nn.Dropout(cfg["dropout"]); self._dropout_modules.append(d)
            layers += [lin, act(), d]; in_dim = h
        self.hidden = nn.Sequential(*layers)
        self.output_layer = NTPLinear(n_ens, in_dim, output_dim)

    def forward(self, x_num, x_cat):
        x_num = x_num.unsqueeze(1).expand(-1, self.n_ens, -1)
        x_cat = x_cat.unsqueeze(1).expand(-1, self.n_ens, -1)
        x = torch.cat([x_num, self.cate(x_cat)], dim=2)
        return F.softmax(self.output_layer(self.hidden(x)), dim=2)

# TabM uses more ensemble members and a different shape than RealMLP.
TABM_VARIANTS = [
    {'seed': 7,  'overrides': {'hidden_dims': [384, 384, 384], 'n_ens': 16, 'dropout': 0.10,
                               'activation': nn.GELU, 'tfms': ['median_center', 'robust_scale', 'smooth_clip']}},
    {'seed': 99, 'overrides': {'hidden_dims': [512, 256, 128], 'n_ens': 16, 'dropout': 0.15,
                               'activation': nn.GELU, 'tfms': ['median_center', 'robust_scale', 'smooth_clip']}},
]
if SMOKE_TEST:
    TABM_VARIANTS = TABM_VARIANTS[:1]
print('TabM enabled:', RUN_TABM, '| variants:', len(TABM_VARIANTS), '| SMOKE_TEST:', SMOKE_TEST)


## 6. RealMLP cross-validation

Default is a **multi-seed** RealMLP average (`MLP_SEEDS = [42, 1337, 2024]`) over the shared stratified folds. Each fold trains with early stopping (`patience`) and keeps its best-epoch checkpoint, so the longer epoch budget is cheap when a fold converges early. Progress is shown with nested tqdm bars (per-seed fold bar + per-epoch bar) and a compact per-fold / per-seed summary table at the end.

In [ ]:
NN_VARIANTS = [{'kind': 'mlp', 'model_cls': RealMLP, **v} for v in MLP_VARIANTS]
if RUN_TABM:
    NN_VARIANTS += [{'kind': 'tabm', 'model_cls': TabM, **v} for v in TABM_VARIANTS]

nn_oof = {}; nn_test = {}; nn_cnt = {}; nn_time = {}
fold_ba_log = []; variant_log = []
oof_tabm = None; test_tabm = None

banner(f'NN cross-validation - {len(NN_VARIANTS)} variant(s) x {N_SPLITS} folds')
t0 = time.time()
for si, variant in enumerate(NN_VARIANTS, 1):
    kind = variant['kind']; seed = variant['seed']; ov = variant['overrides']
    np.random.seed(seed); random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    cfg = {**CONFIG, 'random_state': seed, 'model_cls': variant['model_cls'], **ov}
    shape = cfg['hidden_dims']
    oof_s = np.zeros((len(train_fe), NC), dtype='float32')
    test_s = np.zeros((n_test, NC), dtype='float32')
    tv = time.time()
    fold_bar = tqdm(list(enumerate(folds, 1)),
                    desc=f'[{si}/{len(NN_VARIANTS)}] {kind} seed {seed} {shape}', unit='fold')
    for f, (tr, va) in fold_bar:
        tr_mlp = tr if MLP_USE_EXTERNAL else tr[~is_ext_row[tr]]
        X_tr, X_val, X_tst = X_mlp.iloc[tr_mlp].copy(), X_mlp.iloc[va].copy(), X_mlp_test.copy()
        enc = TargetEncoder(target_type='multiclass', cv=N_SPLITS, smooth='auto',
                            shuffle=True, random_state=SEED)
        tr_e = enc.fit_transform(X_tr[combo_names], y[tr_mlp])
        va_e = enc.transform(X_val[combo_names]); te_e = enc.transform(X_tst[combo_names])
        te_names = [f"_te_{col}_{c}" for col in combo_names for c in range(NC)]
        X_tr[te_names] = tr_e; X_val[te_names] = va_e; X_tst[te_names] = te_e
        model = RealMLP_TD_Classifier(**cfg)
        model.fit(X_tr, y[tr_mlp], X_val, y[va], cat_col_names=mlp_cat_cols,
                  ckpt_path=str(CKPT_DIR / f"{kind}_s{seed}_f{f}.pth"), X_test=X_tst,
                  desc=f'{kind} s{seed} f{f}/{N_SPLITS}')
        oof_s[va] = model.best_val_probs_
        test_s += model.predict_proba(X_tst) / N_SPLITS
        fold_ba_log.append({'variant': si, 'kind': kind, 'seed': seed, 'fold': f,
                            'val_ba': model.best_score_, 'stop_ep': model.stopped_epoch_})
        fold_bar.set_postfix(ba=f'{model.best_score_:.5f}', stop=model.stopped_epoch_)
        del model; torch.cuda.empty_cache(); gc.collect()
    fold_bar.close()
    seed_ba = balanced_accuracy_score(y[real_idx], oof_s[real_idx].argmax(1))
    variant_log.append({'variant': si, 'kind': kind, 'seed': seed,
                        'shape': str(shape), 'oof_ba': seed_ba})
    tqdm.write(f'  >> [{si}] {kind} seed {seed} {shape} OOF BA: {seed_ba:.5f}')
    nn_oof[kind] = nn_oof.get(kind, 0) + oof_s
    nn_test[kind] = nn_test.get(kind, 0) + test_s
    nn_cnt[kind] = nn_cnt.get(kind, 0) + 1
    nn_time[kind] = nn_time.get(kind, 0.0) + (time.time() - tv)

for kind in list(nn_oof):
    nn_oof[kind] /= nn_cnt[kind]; nn_test[kind] /= nn_cnt[kind]
    save_probs(kind, nn_oof[kind], nn_test[kind])
    log_model(kind, nn_oof[kind], nn_time[kind],
              notes=f'{kind} {nn_cnt[kind]}-variant avg; external={MLP_USE_EXTERNAL}')
oof_mlp, test_mlp = nn_oof['mlp'], nn_test['mlp']
if 'tabm' in nn_oof:
    oof_tabm, test_tabm = nn_oof['tabm'], nn_test['tabm']
elapsed = time.time() - t0

pd.DataFrame(variant_log).to_csv(ARTIFACT_DIR / 'nn_variant_metrics.csv', index=False)
pd.DataFrame(fold_ba_log).to_csv(ARTIFACT_DIR / 'nn_fold_metrics.csv', index=False)
banner('NN summary', char='-')
print('per-variant OOF BA:')
display(pd.DataFrame(variant_log).round(5))
for kind in nn_oof:
    kba = balanced_accuracy_score(y[real_idx], nn_oof[kind][real_idx].argmax(1))
    print(f'{kind} {nn_cnt[kind]}-variant avg OOF BA: {kba:.5f}  ({nn_time[kind]:.0f}s)')
print(f'total NN wall time: {elapsed:.0f}s')


## 7. Select final probabilities

Two ensembling strategies over all available models (boosters + RealMLP + TabM) are compared on OOF balanced accuracy and the better one is kept:

- **hill-climb** simplex-weighted blend of probabilities;
- **stacking** — a CV-safe multinomial logistic-regression meta-model on the stacked OOF probabilities.

The winner feeds the per-class multiplier calibration and the submission.

In [ ]:
yv = y[real_idx]

# gather every available model (boosters + NN backbones) into the ensemble pool
model_pool = {}
if 'oof_mlp' in globals():
    model_pool['mlp'] = (oof_mlp, test_mlp)
if 'oof_tabm' in globals() and oof_tabm is not None:
    model_pool['tabm'] = (oof_tabm, test_tabm)
if RUN_BOOSTERS:
    for k in ['lgb', 'xgb', 'cb']:
        if f'oof_{k}' in globals():
            model_pool[k] = (globals()[f'oof_{k}'], globals()[f'test_{k}'])

keys = list(model_pool.keys())
oofs = {k: model_pool[k][0][real_idx] for k in keys}
tests = {k: model_pool[k][1] for k in keys}
print('ensemble members:', keys)
for k in keys:
    print(f'  {k:5s} OOF BA: {balanced_accuracy_score(yv, oofs[k].argmax(1)):.5f}')

# ---- (a) hill-climb weight blend ----------------------------------------
def _blend(ws):
    return sum(w * oofs[k] for w, k in zip(ws, keys))
def _ba(ws):
    return balanced_accuracy_score(yv, _blend(ws).argmax(1))

def hill_climb(active):
    w = np.array([1.0 if k in active else 0.0 for k in keys]); w /= w.sum()
    best = _ba(w); step = 0.05
    for _ in range(800):
        improved = False
        for i, k in enumerate(keys):
            if k not in active:
                continue
            for d in (step, -step):
                cand = w.copy(); cand[i] += d
                if (cand < -1e-9).any():
                    continue
                cand = cand / cand.sum()
                s = _ba(cand)
                if s > best + 1e-6:
                    best, w, improved = s, cand, True
        if not improved:
            break
    return w, best

w_hc, ba_hc = hill_climb(keys)
hc_oof = _blend(w_hc)
hc_test = sum(w * tests[k] for w, k in zip(w_hc, keys))
print('hill-climb blend BA: %.5f  w=%s' % (ba_hc, dict(zip(keys, w_hc.round(3)))))

# ---- (b) stacking meta-model (multinomial logreg, CV-safe) --------------
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
meta_X = np.hstack([oofs[k] for k in keys]).astype('float64')
meta_test = np.hstack([tests[k] for k in keys]).astype('float64')
meta = LogisticRegression(max_iter=3000, C=1.0, class_weight='balanced')
skf_meta = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
stack_oof = cross_val_predict(meta, meta_X, yv, cv=skf_meta, method='predict_proba', n_jobs=1)
ba_stack = balanced_accuracy_score(yv, stack_oof.argmax(1))
meta.fit(meta_X, yv)
stack_test = meta.predict_proba(meta_test)
print('stacking (logreg) BA: %.5f' % ba_stack)

# ---- choose the better of {hill-climb, stack} ---------------------------
if ba_stack >= ba_hc:
    blend_oof, blend_test, blend_name, blend_ba = stack_oof, stack_test, 'stack_logreg', ba_stack
else:
    blend_oof, blend_test, blend_name, blend_ba = hc_oof, hc_test, 'blend_hillclimb', ba_hc
print('SELECTED ->', blend_name, 'BA %.5f' % blend_ba)

RUN_LOG.append(score_row('blend_hillclimb_argmax', hc_oof, notes='hill-climb: ' + ','.join(keys)))
RUN_LOG.append(score_row('stack_logreg_argmax', stack_oof, notes='logreg stack: ' + ','.join(keys)))

blend_meta = {
    'pipeline_mode': PIPELINE_MODE,
    'keys': keys,
    'hillclimb_weights': {k: float(w) for k, w in zip(keys, w_hc)},
    'hillclimb_ba': float(ba_hc),
    'stack_ba': float(ba_stack),
    'selected': blend_name,
}
with open(ARTIFACT_DIR / 'blend_meta.json', 'w') as f:
    json.dump(blend_meta, f, indent=2)
oof_blend_full = np.zeros((len(train_fe), NC), dtype='float32')
oof_blend_full[real_idx] = blend_oof.astype('float32')
save_probs(blend_name + '_argmax', oof_blend_full, blend_test)
display(pd.DataFrame(RUN_LOG).sort_values('oof_balanced_accuracy', ascending=False))


### Class multiplier tuning and diagnostics

Balanced accuracy can improve when class probabilities are rescaled before `argmax`. The same calibrated probabilities feed the OOF diagnostics and final submission.


In [ ]:
_yv_class_idx = [np.where(yv == c)[0] for c in range(NC)]

def _bal_acc_fast(pred):
    """Balanced accuracy with precomputed class index masks (yv is constant)."""
    return float(np.mean([(pred[ix] == c).mean() for c, ix in enumerate(_yv_class_idx)]))

def tune_class_mult(oof_proba, n_rounds=12):
    """Greedy per-class probability multipliers maximising OOF balanced accuracy."""
    mult = np.ones(NC)
    base = _bal_acc_fast(oof_proba.argmax(1))
    for _ in range(n_rounds):
        improved = False
        for c in range(NC):
            for cand in np.linspace(0.5, 2.0, 31):
                wt = mult.copy(); wt[c] = cand
                s = _bal_acc_fast((oof_proba * wt).argmax(1))
                if s > base + 1e-6:
                    base, mult, improved = s, wt, True
        if not improved:
            break
    return mult / mult.mean(), base

def ba_w(proba, mult):
    return balanced_accuracy_score(yv, (proba * mult).argmax(1))

mult, base = tune_class_mult(blend_oof)
print('class multipliers:', dict(zip(CLASSES, mult.round(3))))
print('blend BA argmax  : %.5f' % ba_w(blend_oof, np.ones(NC)))
print('blend BA weighted: %.5f' % base)
print()

final_oof = (blend_oof * mult).argmax(1)
RUN_LOG.append({**score_row(f'{blend_name}_weighted', blend_oof * mult,
                            notes='selected probabilities + per-class multipliers'),
                'class_multipliers': json.dumps({k: float(v) for k, v in zip(CLASSES, mult)})})
print(classification_report(yv, final_oof, target_names=CLASSES, digits=4))
print('confusion matrix (rows=true):')
cm = confusion_matrix(yv, final_oof, labels=np.arange(NC))
cm_df = pd.DataFrame(cm, index=CLASSES, columns=CLASSES)
display(cm_df)

cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(pd.DataFrame(cm_norm, index=CLASSES, columns=CLASSES), annot=True, fmt='.3f', cmap='Blues')
plt.title('Final blend OOF confusion matrix (row-normalized)')
plt.xlabel('predicted'); plt.ylabel('true')
plt.tight_layout()
plt.show()

valid_diag = train_fe.iloc[real_idx][['redshift', 'spectral_type', 'galaxy_population']].copy()
valid_diag['true'] = [int_to_class[i] for i in yv]
valid_diag['pred'] = [int_to_class[i] for i in final_oof]
valid_diag['correct'] = valid_diag['true'].eq(valid_diag['pred'])
valid_diag['error_pair'] = valid_diag['true'] + '->' + valid_diag['pred']
valid_diag['redshift_bin'] = pd.cut(valid_diag['redshift'],
                                    [-np.inf, 0, .01, .05, .1, .25, .5, np.inf])

err = valid_diag[~valid_diag['correct']]
star_gal_err = err[err['error_pair'].isin(['STAR->GALAXY', 'GALAXY->STAR'])]
print('Errors:', len(err), '| STAR/GALAXY boundary errors:', len(star_gal_err))
display(star_gal_err.groupby(['error_pair', 'redshift_bin'], observed=True).size()
        .unstack(fill_value=0))
display(star_gal_err.groupby(['error_pair', 'spectral_type', 'galaxy_population'], observed=True)
        .size().sort_values(ascending=False).head(20).rename('n').to_frame())

fig, ax = plt.subplots(figsize=(9, 4))
star_gal_err.groupby(['redshift_bin', 'error_pair'], observed=True).size().unstack(fill_value=0).plot(kind='bar', ax=ax)
ax.set_title('STAR/GALAXY OOF errors by redshift bin')
ax.set_xlabel('redshift bin'); ax.set_ylabel('OOF error count')
plt.tight_layout()
plt.show()

np.save(ARTIFACT_DIR / 'oof_blend.npy', (blend_oof * mult).astype('float32'))
pd.DataFrame(RUN_LOG).to_csv(ARTIFACT_DIR / 'experiment_summary.csv', index=False)
pd.DataFrame(FOLD_LOG).to_csv(ARTIFACT_DIR / 'fold_metrics.csv', index=False)
valid_diag.to_csv(ARTIFACT_DIR / 'final_oof_diagnostics.csv', index=False)
with open(ARTIFACT_DIR / 'class_multipliers.json', 'w') as f:
    json.dump({k: float(v) for k, v in zip(CLASSES, mult)}, f, indent=2)
print('Saved diagnostics to', ARTIFACT_DIR.resolve())


## 8. Blend submissions

Several **blend** candidates are emitted, differing in how model probabilities are aggregated: linear stack (logreg), nonlinear stack (LGBM), hill-climb weights, arithmetic / geometric mean, rank-average, and a two-tier NN+GBM blend. Each gets its own per-class multiplier calibration, every candidate is written as `submission_<name>.csv`, and a ranked board shows the order to push. `submission.csv` = best blend by OOF.

In [ ]:
# Blend-focused submissions. Every candidate is an ENSEMBLE; they differ only in
# HOW the model probabilities are aggregated. Improvements over a naive dump:
#   * rank by RAW OOF BA (calibration is tuned on the same OOF -> optimistic; we
#     still SHIP the calibrated file, but pick/order by the less-overfit metric);
#   * de-duplicate: near-identical label vectors are flagged so daily submissions
#     are spent on genuinely DIFFERENT blends, not 7 copies of the same one;
#   * a 'pct_diff_vs_canon' column shows how distinct each candidate really is.
import shutil

def _norm(p):
    return p / np.clip(p.sum(1, keepdims=True), 1e-12, None)

def sub_from_labels(lab):
    return pd.DataFrame({ID: test_fe[ID].values, TARGET: [int_to_class[i] for i in lab]})

def geom_mean(plist):
    L = sum(np.log(np.clip(p, 1e-9, 1)) for p in plist) / len(plist)
    return _norm(np.exp(L))

def _rank_cols(P):
    R = np.empty_like(P, dtype='float64')
    for c in range(P.shape[1]):
        order = P[:, c].argsort()
        r = np.empty(len(order), dtype='float64'); r[order] = np.arange(len(order))
        R[:, c] = r / max(len(order) - 1, 1)
    return R

def rank_blend(plist):
    return sum(_rank_cols(p) for p in plist) / len(plist)

nn_keys = [k for k in ['mlp', 'tabm'] if k in keys]
gbm_keys = [k for k in ['lgb', 'xgb', 'cb'] if k in keys]

# --- candidate pool: blends only ---
candidates = {}
candidates['stack_logreg'] = (stack_oof, stack_test)
candidates['hillclimb'] = (hc_oof, hc_test)
candidates['mean_arith'] = (sum(oofs[k] for k in keys) / len(keys),
                            sum(tests[k] for k in keys) / len(keys))
candidates['mean_geom'] = (geom_mean([oofs[k] for k in keys]),
                           geom_mean([tests[k] for k in keys]))
candidates['rank_mean'] = (rank_blend([oofs[k] for k in keys]),
                           rank_blend([tests[k] for k in keys]))
try:
    import lightgbm as lgb
    from sklearn.model_selection import cross_val_predict
    meta_lgb = lgb.LGBMClassifier(objective='multiclass', num_class=NC, n_estimators=400,
                                  learning_rate=0.03, num_leaves=31, max_depth=4,
                                  subsample=0.8, colsample_bytree=0.8, min_child_samples=100,
                                  reg_lambda=2.0, random_state=SEED, n_jobs=-1, verbose=-1)
    with quiet():
        sl_oof = cross_val_predict(meta_lgb, meta_X, yv,
                                   cv=StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED),
                                   method='predict_proba', n_jobs=1)
        meta_lgb.fit(meta_X, yv); sl_test = meta_lgb.predict_proba(meta_test)
    candidates['stack_lgbm'] = (sl_oof, sl_test)
except Exception as e:
    print('stack_lgbm skipped:', repr(e))
if nn_keys and gbm_keys:
    wnn, _ = hill_climb(nn_keys); wg, _ = hill_climb(gbm_keys)
    nn_o, nn_t = _blend(wnn), sum(w * tests[k] for w, k in zip(wnn, keys))
    g_o, g_t = _blend(wg), sum(w * tests[k] for w, k in zip(wg, keys))
    candidates['nn_plus_gbm'] = (0.5 * nn_o + 0.5 * g_o, 0.5 * nn_t + 0.5 * g_t)

# --- score + calibrate each candidate ONCE; cache mult + calibrated labels ---
info = {}
for name, (o, t) in candidates.items():
    m, ba_cal = tune_class_mult(o)
    info[name] = {'mult': m, 'oof_ba_raw': balanced_accuracy_score(yv, o.argmax(1)),
                  'oof_ba_cal': ba_cal, 'labels': (t * m).argmax(1)}

# canonical = best by RAW OOF BA (robust to calibration selection bias)
order = sorted(info, key=lambda n: info[n]['oof_ba_raw'], reverse=True)
canon = order[0]; canon_lab = info[canon]['labels']

# de-duplicate by test-label agreement (keep highest-raw representative)
kept, dup_of = [], {}
for n in order:
    lab = info[n]['labels']
    d = next((kn for kn in kept if (lab != info[kn]['labels']).mean() < 5e-4), None)
    if d is None:
        kept.append(n)
    else:
        dup_of[n] = d

# --- write every candidate once, copy to artifacts ---
for name in candidates:
    sd = sub_from_labels(info[name]['labels'])
    assert list(sd.columns) == [ID, TARGET] and len(sd) == len(sample_sub)
    fn = f'submission_{name}.csv'
    sd.to_csv(OUT_DIR / fn, index=False); shutil.copy(OUT_DIR / fn, ARTIFACT_DIR / fn)

# canonical (calibrated) + uncalibrated probe of the same blend
sub_from_labels(info[canon]['labels']).to_csv(OUT_DIR / 'submission.csv', index=False)
raw_fn = f'submission_{canon}_raw.csv'
sub_from_labels(candidates[canon][1].argmax(1)).to_csv(OUT_DIR / raw_fn, index=False)
shutil.copy(OUT_DIR / raw_fn, ARTIFACT_DIR / raw_fn)

board = pd.DataFrame([{
    'candidate': n, 'distinct': n in kept, 'dup_of': dup_of.get(n, ''),
    'oof_ba_raw': round(info[n]['oof_ba_raw'], 5), 'oof_ba_cal': round(info[n]['oof_ba_cal'], 5),
    'pct_diff_vs_canon': round(100 * (info[n]['labels'] != canon_lab).mean(), 3),
    'file': f'submission_{n}.csv',
} for n in order])
board.to_csv(ARTIFACT_DIR / 'submission_board.csv', index=False)
save_run_logs()

banner('Blend submissions')
display(board)
n_distinct = int(board['distinct'].sum())
print(f'canonical submission.csv = submission_{canon}.csv  (best RAW OOF = {info[canon]["oof_ba_raw"]:.5f}, '
      f'cal = {info[canon]["oof_ba_cal"]:.5f})')
print(f'overfit probe            = {raw_fn} (uncalibrated)')
print(f'\n{n_distinct} DISTINCT blends (distinct=True). Push those + the _raw probe; '
      f'skip rows with a dup_of (near-identical labels, <0.05% diff).')
print('rank by oof_ba_raw, but trust the LB over OOF near the ceiling.')
